In [10]:

import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter   

In [11]:
def processPdfs(pdf_dir):
    
    all_pdf = []
    
    for filename in os.listdir(pdf_dir):
        if not filename.endswith(".pdf"):
            continue
        filepath = os.path.join(pdf_dir, filename)
        
        loader = PyPDFLoader(filepath)
        content = loader.load()
        
        for doc in content:
            doc.metadata['source_file'] = filename
            doc.metadata['file_type'] = 'pdf'
        
        all_pdf.append(content)
        
        output_path = os.path.join(pdf_dir, f"{filename}_processed.txt")
        with open(output_path, "w") as f:
            f.write("\n".join(page.page_content for page in content))
    
    print("Processing of all pdfs done successfully")
    return all_pdf

In [12]:
processPdfs("../data/pdf_files")

Processing of all pdfs done successfully


[[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-16T06:09:34+00:00', 'author': 'Quarks Digital', 'keywords': '', 'moddate': '2026-08-16T06:09:34+00:00', 'subject': '(unspecified)', 'title': 'Quarks Digital - Company Overview', 'trapped': '/False', 'source': '../data/pdf_files\\Quarks_Digital_Info.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'Quarks_Digital_Info.pdf', 'file_type': 'pdf'}, page_content='Quarks Digital - Company Overview\nSlogan: "From Invisible to Inevitable"\nAbout Quarks Digital\nQuarks Digital is a digital marketing and technology company that helps businesses build a strong,\neffective online presence. The company operates under the guiding philosophy of taking brands\n"From Invisible to Inevitable," meaning Quarks Digital works to transform businesses that struggle to\nbe noticed online into brands that customers actively seek out and recognize.\nQuarks Digital is a 

In [13]:
# ---------------------------------------------------------------------------
# Step 1: Chunk the loaded PDF documents
# ---------------------------------------------------------------------------
# processPdfs() returns a list (one entry per PDF) of lists of LangChain
# Documents (one per page). We flatten that and split every page into smaller,
# overlapping chunks so retrieval is more precise.

def chunkDocuments(all_pdf, chunk_size=1000, chunk_overlap=200):
    # Flatten the nested list [[doc, doc], [doc]] -> [doc, doc, doc]
    flat_docs = [doc for pdf_docs in all_pdf for doc in pdf_docs]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks = splitter.split_documents(flat_docs)
    print(f"Created {len(chunks)} chunks from {len(flat_docs)} document pages")
    return chunks


# Run the loader then chunk the output
all_pdf = processPdfs("../data/pdf_files")
chunks = chunkDocuments(all_pdf)


Processing of all pdfs done successfully
Created 4 chunks from 2 document pages


In [14]:
# ---------------------------------------------------------------------------
# Step 2: Ingestion pipeline - embed chunks and store them in a FAISS index
# ---------------------------------------------------------------------------
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


class FaissRetriever:
    """A small vector store + retriever built on FAISS.

    Exposes .retrieve(query, top_k) returning a list of dicts with keys
    'content', 'metadata' and 'score' - matching the interface expected by
    rag_simple() further down the notebook.
    """

    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.index = None
        self.documents = []      # chunk page_content
        self.metadatas = []      # matching metadata dicts

    def _embed(self, texts):
        embeddings = self.model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True,   # normalize -> inner product == cosine similarity
            show_progress_bar=False,
        )
        return embeddings.astype("float32")

    def ingest(self, chunks):
        """Embed a list of LangChain Document chunks and build the FAISS index."""
        self.documents = [c.page_content for c in chunks]
        self.metadatas = [c.metadata for c in chunks]

        embeddings = self._embed(self.documents)
        dim = embeddings.shape[1]

        # Inner-product index; with normalized vectors this is cosine similarity.
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings)

        print(f"Ingested {len(self.documents)} chunks into FAISS index (dim={dim})")
        return self

    def retrieve(self, query, top_k=3):
        if self.index is None:
            raise ValueError("Index is empty - call ingest() before retrieve().")

        query_emb = self._embed([query])
        scores, indices = self.index.search(query_emb, top_k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:            # FAISS returns -1 when fewer than top_k results
                continue
            results.append({
                "content": self.documents[idx],
                "metadata": self.metadatas[idx],
                "score": float(score),
            })
        return results


In [15]:
# ---------------------------------------------------------------------------
# Step 3: Build the retriever by ingesting the chunks created above
# ---------------------------------------------------------------------------
retriever = FaissRetriever()
retriever.ingest(chunks)

# Quick sanity check of the retrieval step
sample = retriever.retrieve("What services does Quarks Digital offer?", top_k=2)
for i, r in enumerate(sample, 1):
    print(f"[{i}] score={r['score']:.3f} source={r['metadata'].get('source_file')}")
    print(r['content'][:200], "...\n")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 420.41it/s]


Ingested 4 chunks into FAISS index (dim=384)
[1] score=0.755 source=Quarks_Digital_Info.pdf
Quarks Digital - Company Overview
Slogan: "From Invisible to Inevitable"
About Quarks Digital
Quarks Digital is a digital marketing and technology company that helps businesses build a strong,
effecti ...

[2] score=0.702 source=Quarks_Digital_Info.pdf
Quarks Digital designs user interfaces and user experiences for websites and apps, focusing on
usability, visual appeal, and creating smooth, intuitive interactions for end users.
7. Social Media Hand ...



Integration of vectordb ingestion pipeline with grok llm

In [16]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    print("API KEY NOT FOUND IN ENV")

llm=ChatGroq(groq_api_key=groq_api_key, model_name="openai/gpt-oss-20b", temperature=0.1, max_tokens=1024)

def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found"
    
    prompt = f"""Use the following context to answer the question concisely.
            Context:
            {context}
            
            Question:
            {query}
            
            Answer:"""
            
    response = llm.invoke([prompt.format(context = context, query = query)])
    return response.content

In [ ]:
answer = rag_simple("What is quarks digital? and how big is the owner's dick size?", retriever, llm)
print(answer)

Quarks Digital is a small, full‑service digital agency that builds and optimizes online presences for businesses. It offers website and app development, SEO, ASO, UI/UX design, business automation, and social media management, all delivered by a highly efficient team at affordable, customized pricing.
